# 01 - Geracao de Mascaras e Segmentacao

Processa imagens usando modelos rembg para gerar máscaras de segmentação em escala de cinza (raw).

## Imports

Carrega bibliotecas necessárias para processamento de imagens e segmentação com rembg.

In [ ]:
import sys
from pathlib import Path

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import gc
import os
import time

import cv2
import numpy as np
from PIL import Image

from src.io.indice_loader import carregar_indice_excel
from src.io.path_utils import caminho_foto_original, caminho_ground_truth, caminho_mascara_predita
from rembg import new_session, remove
from src.runtime.runtime_config import ORT_PROVIDERS, available_providers, resolver_providers, use_cuda
from src.logs.segmentacao_logs import imprimir_resumo_modelo, imprimir_status

if use_cuda:
    print('GPU NVIDIA ativa: usando CUDAExecutionProvider com fallback para CPU.')
else:
    print('GPU NVIDIA/CUDA indisponivel: usando somente CPUExecutionProvider.')

print(f'Providers disponiveis no ambiente: {available_providers}')
print(f'Providers configurados para a sessao: {ORT_PROVIDERS}')


## Configuração

Carrega configurações do projeto (diretórios e modelos para avaliação).

In [ ]:
from src.config import (
    GENERATED_DIR,
    MODELOS_PARA_AVALIACAO,
    PREDICTED_MASKS_DIR,
)

## Leitura do índice

Carrega o arquivo Excel com o índice das imagens a serem processadas.

In [ ]:
indice_excel = carregar_indice_excel()

## Verificação de integridade dos PNGs gerados

Valida arquivos PNG em `generated/` e remove imagens corrompidas detectadas.

In [ ]:
# Verifica integridade dos PNGs em generated/ e remove arquivos corrompidos
def verificar_e_limpar_pngs_corrompidos(diretorio_base):
    total_png = 0
    arquivos_integros = 0
    arquivos_removidos = 0
    falhas_remocao = 0

    for raiz, _, arquivos in os.walk(diretorio_base):
        for nome_arquivo in arquivos:
            if not nome_arquivo.lower().endswith('.png'):
                continue

            total_png += 1
            caminho_arquivo = os.path.join(raiz, nome_arquivo)

            try:
                with Image.open(caminho_arquivo) as img:
                    img.verify()

                with Image.open(caminho_arquivo) as img:
                    img.load()

                arquivos_integros += 1
            except Exception as e:
                print(f"Corrompido detectado: {caminho_arquivo}")
                print(f" - Erro: {e}")

                try:
                    os.remove(caminho_arquivo)
                    arquivos_removidos += 1
                    print(" - Arquivo removido com sucesso.")
                except OSError as erro_remocao:
                    falhas_remocao += 1
                    print(f" - Falha ao remover arquivo: {erro_remocao}")

    print('\nVerificacao de integridade concluida.')
    print(f" - Total de PNGs verificados: {total_png}")
    print(f" - Arquivos integros: {arquivos_integros}")
    print(f" - Arquivos removidos por corrupcao: {arquivos_removidos}")
    print(f" - Falhas ao remover: {falhas_remocao}")

verificar_e_limpar_pngs_corrompidos(GENERATED_DIR)

## Processa as imagens e gera máscaras raw

Executa os modelos de segmentação configurados e gera máscaras em escala de cinza (grayscale 0-255) sem binarização.

### Configuração do rembg

- `only_mask=True`: Retorna apenas a máscara de segmentação

### Output

Máscaras salvas em: `generated/predicted_masks/<modelo>/*.png`

In [ ]:
def log_status(stats_geral, stats_modelo, nome_modelo):
    tempo_geral = time.perf_counter() - stats_geral['inicio']
    tempo_modelo = time.perf_counter() - stats_modelo['inicio']
    imprimir_status(stats_geral, stats_modelo, nome_modelo, tempo_geral, tempo_modelo)


total_modelos = len(MODELOS_PARA_AVALIACAO)
total_imagens = len(indice_excel)
total_previsto = total_modelos * total_imagens

stats_geral = {
    'total': total_previsto,
    'processadas': 0,
    'ok': 0,
    'skip': 0,
    'erro': 0,
    'inicio': time.perf_counter(),
    'tempo_inferencia': 0.0
}

for model, provider_config in MODELOS_PARA_AVALIACAO.items():
    providers = resolver_providers(provider_config, model)
    print(f'Iniciando modelo: {model} (provider: {provider_config})')

    stats_modelo = {
        'total': total_imagens,
        'processadas': 0,
        'ok': 0,
        'skip': 0,
        'erro': 0,
        'inicio': time.perf_counter(),
        'tempo_inferencia': 0.0
    }

    try:
        rembg_session = new_session(model, providers=providers)
    except Exception as e:
        print(f" - Falha ao iniciar sessao com providers {providers}: {e}")
        print(' - Recriando sessao em CPU...')
        rembg_session = new_session(model, providers=['CPUExecutionProvider'])

    output_dir = os.path.join(PREDICTED_MASKS_DIR, model)
    os.makedirs(output_dir, exist_ok=True)

    for linha in indice_excel:
        original_path = caminho_foto_original(linha.nome_arquivo)
        mascara_path = caminho_ground_truth(linha.nome_arquivo)
        output_path = caminho_mascara_predita(model, linha.nome_arquivo)

        if not os.path.isfile(original_path):
            print(f"[ERRO ] Arquivo original nao encontrado: {original_path}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            log_status(stats_geral, stats_modelo, model)
            continue

        if not os.path.isfile(mascara_path):
            print(f"[ERRO ] Arquivo de mascara nao encontrado: {mascara_path}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            log_status(stats_geral, stats_modelo, model)
            continue

        if os.path.isfile(output_path):
            stats_geral['processadas'] += 1
            stats_geral['skip'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['skip'] += 1
            log_status(stats_geral, stats_modelo, model)
            continue

        try:
            inicio_inferencia = time.perf_counter()
            with Image.open(original_path) as input_rembg:
                output_rembg = remove(
                    input_rembg,
                    only_mask=True,
                    session=rembg_session
                )
            output_rembg.save(output_path)

            duracao_inferencia = time.perf_counter() - inicio_inferencia
            stats_geral['processadas'] += 1
            stats_geral['ok'] += 1
            stats_geral['tempo_inferencia'] += duracao_inferencia

            stats_modelo['processadas'] += 1
            stats_modelo['ok'] += 1
            stats_modelo['tempo_inferencia'] += duracao_inferencia

            log_status(stats_geral, stats_modelo, model)
        except Exception as e:
            print(f"[ERRO ] Falha ao segmentar {linha.nome_arquivo} no modelo {model}: {e}")
            stats_geral['processadas'] += 1
            stats_geral['erro'] += 1
            stats_modelo['processadas'] += 1
            stats_modelo['erro'] += 1
            log_status(stats_geral, stats_modelo, model)

    tempo_modelo = time.perf_counter() - stats_modelo['inicio']
    imprimir_resumo_modelo(model, stats_modelo, tempo_modelo)

